# Experiment 7 — Operational threshold evaluation (PhreshPhish, URL-only LightGBM)

**Question.** Experiment 4 published a PhreshPhish temporal result at a fixed
0.5 threshold. A deployed filter cannot be operated at 0.5 if that costs
6.1 % false alarms on benign URLs. This notebook asks what the model is worth
when the operating point is *chosen* — and chooses it without ever looking at
the test period.

**Protocol.**

| block | dates | role |
|---|---|---|
| fit | 2024-07-02 .. 2025-06-30 | training only |
| validation | 2025-07-01 .. 2025-09-07 | threshold selection only |
| test | 2025-09-08 .. 2025-12-16 | one evaluation, thresholds already frozen |

Cleaning (seam purge, URL canonicalisation, duplicate removal, train→test URL
purge) is Experiment 4's, applied in the same order, so the test partition here
is row-for-row Experiment 4's. The model parameters are Experiment 4's,
untouched: this experiment selects operating points, it does not tune models.

**Pre-declared selection rules** (fixed before any score exists):

1. `default_0p5` — the 0.5 rule Experiment 4 published, kept as the reference.
2. `fpr_le_1pct` — smallest validation threshold whose benign FPR ≤ 1 %.
3. `fpr_le_0p5pct` — same with a 0.5 % cap.
4. `max_val_f1` — validation phishing-positive F1 maximum (ties → smaller threshold).

Rules 2 and 3 maximise recall subject to an alarm budget, because benign FPR is
non-increasing in the threshold.

In [1]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config as C

print("python  ", sys.version.split()[0])
from src.run_experiment import _versions
for name, version in _versions().items():
    print(f"{name:14s}{version}")
print("fit       ", "< VAL_START", C.VAL_START)
print("validation", C.VAL_START, ".. <", C.CUTOFF)
print("test      ", ">= ", C.CUTOFF)
print("seeds     ", C.SEEDS)

python   3.14.7


numpy         2.5.3
pandas        3.1.0
pyarrow       26.0.0
scikit-learn  1.9.1
lightgbm      4.7.0
tldextract    5.4.0
fit        < VAL_START 2025-07-01 00:00:00+00:00
validation 2025-07-01 00:00:00+00:00 .. < 2025-09-08 00:00:00+00:00
test       >=  2025-09-08 00:00:00+00:00
seeds      [13, 42, 2024, 777, 31415]


## 1. Static checks

Threshold logic is checked on synthetic data with hand-computed answers before
any corpus is touched: selection rules must obey their stated constraint, the
vectorised cumulative-count curve must equal a brute-force recount, and a
single-class validation block must raise instead of picking something.

In [2]:
from src.selfcheck import check_threshold_logic, check_curve_matches_bruteforce

check_threshold_logic()
check_curve_matches_bruteforce()
print("threshold selection logic verified on synthetic data")

threshold selection logic verified on synthetic data


## 2. Fit, score, freeze, evaluate

`run()` performs the whole study in the order that makes the leakage argument
checkable: load → clean → split → featurise → fit on the fit block → score
validation and test → **freeze thresholds on validation** → evaluate those
frozen thresholds on test once. Nothing after the freeze reads validation
labels again, and nothing before it reads test labels.

Every artifact lands in a timestamped `outputs/<utc>/` directory: shard and
output manifests with SHA-256, the row ledger, all three split files, per-seed
validation and test probabilities (enough to recompute every metric
independently), the saved LightGBM boosters, `thresholds_frozen.csv`,
`test_metrics.csv` and `ppv_projections.csv`.

In [3]:
from src.run_experiment import run

out_dir = run()
print(out_dir)

Output: E:\Kuliah\Tugas Akhir\Experiment\7 - Operational Threshold Evaluation\outputs\20261026T064137272536Z


loaded train=498255 test=168060 in 66.4s


                        stage   rows  benign  phish                  min_date                  max_date
                    train_raw 498255  276729 221526 2024-07-02 00:00:00+00:00 2025-09-08 00:00:00+00:00
                     test_raw 168060   91260  76800 2025-09-08 00:00:00+00:00 2025-12-16 00:00:00+00:00
   train_after_boundary_purge 497574  276316 221258 2024-07-02 00:00:00+00:00 2025-09-07 00:00:00+00:00
train_after_unparseable_purge 497573  276316 221257 2024-07-02 00:00:00+00:00 2025-09-07 00:00:00+00:00
                 train_unique 497359  276315 221044 2024-07-02 00:00:00+00:00 2025-09-07 00:00:00+00:00
    test_after_seen_url_purge 168057   91259  76798 2025-09-08 00:00:00+00:00 2025-12-16 00:00:00+00:00
                  test_unique 168035   91259  76776 2025-09-08 00:00:00+00:00 2025-12-16 00:00:00+00:00
                          fit 372213  181762 190451 2024-07-02 00:00:00+00:00 2025-06-30 00:00:00+00:00
                   validation 125146   94553  30593 2025-07-01 0

features (372213, 18) in 31.9s


fitting per seed on the fit block only ...


...

         rule  threshold  f1_phish  precision_phish  recall_phish  fpr_benign  miss_rate      mcc  false_alarms_per_10k_benign
  default_0p5   0.500000  0.806999         0.890301      0.737952    0.076497   0.262048 0.679280                   764.965647
  fpr_le_1pct   0.884705  0.698557         0.975796      0.543998    0.011352   0.456002 0.608973                   113.523050
fpr_le_0p5pct   0.937870  0.641062         0.985569      0.475018    0.005851   0.524982 0.563984                    58.514777
   max_val_f1   0.489218  0.807150         0.886566      0.740791    0.079740   0.259209 0.677811                   797.400804



across seeds (mean):


               f1_phish  precision_phish  recall_phish  fpr_benign  miss_rate  false_alarms_per_10k_benign       mcc
rule                                                                                                                
default_0p5    0.806293         0.890503      0.736634    0.076203   0.263366                   762.028951  0.678496
fpr_le_0p5pct  0.646167         0.985751      0.480645    0.005843   0.519355                    58.427114  0.568392
fpr_le_1pct    0.700739         0.975881      0.546658    0.011365   0.453342                   113.654544  0.611036
max_val_f1     0.806558         0.885146      0.740833    0.080926   0.259167                   809.257169  0.676450



Done in 196.448s.


E:\Kuliah\Tugas Akhir\Experiment\7 - Operational Threshold Evaluation\outputs\20261026T064137272536Z


## 3. Frozen thresholds and what they cost on the test period

Read this table as: each row's threshold was chosen using validation labels
only, then applied unchanged to the untouched test period.

In [4]:
OUT = Path(out_dir)
frozen = pd.read_csv(OUT / "thresholds_frozen.csv")
primary = pd.read_csv(OUT / "test_metrics_primary_seed.csv")
print("validation selection (primary seed):")
display(frozen[frozen.seed == C.PRIMARY_SEED][[
    "rule", "threshold", "val_fpr", "val_recall", "val_precision", "val_f1",
    "val_false_alarms_per_10k_benign"]].round(4))
print("test outcome at the frozen thresholds (primary seed):")
display(primary[[
    "rule", "threshold", "f1_phish", "precision_phish", "recall_phish",
    "fpr_benign", "miss_rate", "false_alarms_per_10k_benign", "mcc",
    "tn", "fp", "fn", "tp"]].round(4))

validation selection (primary seed):


,rule,threshold,val_fpr,val_recall,val_precision,val_f1,val_false_alarms_per_10k_benign
0,default_0p5,0.5000,0.0697,0.6187,0.7417,0.6746,697.2809
1,fpr_le_1pct,0.8847,0.0100,0.4254,0.9324,0.5842,99.7324
2,fpr_le_0p5pct,0.9379,0.0050,0.3822,0.9612,0.5469,49.9191
3,max_val_f1,0.4892,0.0725,0.6245,0.7360,0.6757,724.7787


test outcome at the frozen thresholds (primary seed):


,rule,threshold,f1_phish,precision_phish,recall_phish,fpr_benign,miss_rate,false_alarms_per_10k_benign,mcc,tn,fp,fn,tp
0,default_0p5,0.5000,0.8070,0.8903,0.7380,0.0765,0.2620,764.9656,0.6793,84278,6981,20119,56657
1,fpr_le_1pct,0.8847,0.6986,0.9758,0.5440,0.0114,0.4560,113.5230,0.6090,90223,1036,35010,41766
2,fpr_le_0p5pct,0.9379,0.6411,0.9856,0.4750,0.0059,0.5250,58.5148,0.5640,90725,534,40306,36470
3,max_val_f1,0.4892,0.8071,0.8866,0.7408,0.0797,0.2592,797.4008,0.6778,83982,7277,19901,56875


## 4. Across the five seeds

Seed variation is *not* temporal uncertainty: five seeds are five fits on the
same split, not five time periods.

In [5]:
across = pd.read_csv(OUT / "test_metrics_across_seeds.csv", header=[0, 1], index_col=0)
display(across.round(4))

f1_phish                         precision_phish          \
                  mean     std     min     max            mean     std   
rule                                                                     
default_0p5     0.8063  0.0005  0.8059  0.8070          0.8905  0.0007   
fpr_le_0p5pct   0.6462  0.0081  0.6375  0.6584          0.9858  0.0004   
fpr_le_1pct     0.7007  0.0068  0.6939  0.7117          0.9759  0.0003   
max_val_f1      0.8066  0.0004  0.8061  0.8072          0.8851  0.0064   

                              recall_phish          ... miss_rate          \
                  min     max         mean     std  ...       min     max   
rule                                                ...                     
default_0p5    0.8895  0.8911       0.7366  0.0010  ...    0.2620  0.2645   
fpr_le_0p5pct  0.9853  0.9863       0.4806  0.0089  ...    0.5059  0.5288   
fpr_le_1pct    0.9756  0.9764       0.5467  0.0083  ...    0.4400  0.4617   
max_val_f1     0.8746  0.8915       0.7408  0.0044  ...    0.2522  0.2638   

              false_alarms_per_10k_benign                               \
                                     mean      std       min       max   
rule                                                                     
default_0p5                      762.0290   5.8844  756.0898  770.4446   
fpr_le_0p5pct                     58.4271   0.6638   57.5286   59.2818   
fpr_le_1pct                      113.6545   1.2359  112.3177  115.6050   
max_val_f1                       809.2572  55.9023  754.1174  901.9384   

                  mcc                          
                 mean     std     min     max  
rule                                           
default_0p5    0.6785  0.0007  0.6779  0.6793  
fpr_le_0p5pct  0.5684  0.0071  0.5608  0.5790  
fpr_le_1pct    0.6110  0.0064  0.6047  0.6214  
max_val_f1     0.6764  0.0029  0.6714  0.6791  

[4 rows x 28 columns]

## 5. PPV at deployment prevalence

The test period is ~45 % phishing, which no mail gateway sees. PPV is projected
with the Bayes form `pi·TPR / (pi·TPR + (1-pi)·FPR)` at 1 % and 5 % prevalence,
holding the measured TPR and FPR fixed. That transfer is the assumption, not a
measurement — it is labelled as a projection in the artifact, not a result.

In [6]:
proj = pd.read_csv(OUT / "ppv_projections.csv")
display(proj[proj.seed == C.PRIMARY_SEED][[
    "rule", "threshold", "assumed_prevalence", "test_recall", "test_fpr",
    "projected_ppv", "projected_false_alarms_per_10k_benign"]].round(4))

,rule,threshold,assumed_prevalence,test_recall,test_fpr,projected_ppv,projected_false_alarms_per_10k_benign
0,default_0p5,0.5000,0.01,0.7380,0.0765,0.0888,757.3160
1,default_0p5,0.5000,0.05,0.7380,0.0765,0.3368,726.7174
2,fpr_le_1pct,0.8847,0.01,0.5440,0.0114,0.3262,112.3878
3,fpr_le_1pct,0.8847,0.05,0.5440,0.0114,0.7161,107.8469
4,fpr_le_0p5pct,0.9379,0.01,0.4750,0.0059,0.4505,57.9296
5,fpr_le_0p5pct,0.9379,0.05,0.4750,0.0059,0.8103,55.5890
6,max_val_f1,0.4892,0.01,0.7408,0.0797,0.0858,789.4268
7,max_val_f1,0.4892,0.05,0.7408,0.0797,0.3284,757.5308


## 6. Lineage proof

Two claims need evidence, not assertion: that the test partition is the one
Experiment 4 published, and that the features are Experiment 5's audited A2
extractor. This cell re-reads the raw shards and checks both.

In [7]:
from src.selfcheck import check_lineage

check_lineage()

lineage: test partition identical to Experiment 4 (168035 rows)
lineage: 700 URLs featurised identically to Experiment 5's extractor
                        stage   rows  benign  phish                  min_date                  max_date
                    train_raw 498255  276729 221526 2024-07-02 00:00:00+00:00 2025-09-08 00:00:00+00:00
                     test_raw 168060   91260  76800 2025-09-08 00:00:00+00:00 2025-12-16 00:00:00+00:00
   train_after_boundary_purge 497574  276316 221258 2024-07-02 00:00:00+00:00 2025-09-07 00:00:00+00:00
train_after_unparseable_purge 497573  276316 221257 2024-07-02 00:00:00+00:00 2025-09-07 00:00:00+00:00
                 train_unique 497359  276315 221044 2024-07-02 00:00:00+00:00 2025-09-07 00:00:00+00:00
    test_after_seen_url_purge 168057   91259  76798 2025-09-08 00:00:00+00:00 2025-12-16 00:00:00+00:00
                  test_unique 168035   91259  76776 2025-09-08 00:00:00+00:00 2025-12-16 00:00:00+00:00
                          fit 37221

## 7. Manifest

Every output file hashed into `output_manifest.json`; every input shard hashed
into `input_manifest.json`. `HANDOFF.md` records the rerun command, the runtime
and the limitations.

In [8]:
manifest = pd.DataFrame(json.loads((OUT / "output_manifest.json").read_text()))
print(f"{len(manifest)} files, {manifest.bytes.sum()/1e6:.1f} MB")
display(manifest.assign(mb=(manifest.bytes / 1e6).round(3))[["file", "mb", "sha256"]])

34 files, 74.8 MB


,file,mb,sha256
0,design.json,0.004,a4af9c764f66224e1a0b6545876a81f88ca3701be8fc57...
1,excluded_rows.csv,0.049,7aa0faa37b883f28d59870f229c71297b01128e27672d9...
2,feature_schema.json,0.001,ddfc06887254c74909781f2a2404030abc962e2f083e27...
3,input_manifest.json,0.013,d400944c98ec67f5cc56c43d0afa67c50512576fba9f32...
4,model_params_seed_13.json,0.000,cab3bc147007ee3a3d8674c290dced5514a2209e55a3d3...
5,model_params_seed_2024.json,0.000,d83b0855b28c80516754602ec050ac92caf67c5e1acf10...
6,model_params_seed_31415.json,0.000,059ba1ce28d19cd1cd916d5f171a59b66f45b5329d21ee...
7,model_params_seed_42.json,0.000,7979f485e959a9f75e6151632b0be14e2725624ab9baf8...
8,model_params_seed_777.json,0.000,982ecb1db5f0cee8492bf4cae0ab3504706b3d6ac8c8ea...
9,model_seed_13.txt,2.079,ff3d6817028af32c7ab0f4d15a56f2250eb1ee4dcf2ac4...
